<a href="https://colab.research.google.com/github/Nightwing-77/TensorTonic-Solutions/blob/main/Prefix_sum_kernel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

Sun Jun  7 11:33:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install nvcc4jupyter
%load_ext nvcc4jupyter

Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmptxteb92e".


In [3]:
%%writefile prefix_sum.cu

#include <iostream>
#include <cuda_runtime.h>

__global__ void PartialSumKernel(int *input, int *output, int n) {
    extern __shared__ int sharedMemory[];

    int tid = threadIdx.x;
    int index = blockIdx.x * blockDim.x + tid;

    if (index < n) {
        sharedMemory[tid] = input[index];
    } else {
        sharedMemory[tid] = 0;
    }

    __syncthreads();

    // Inclusive scan within block
    for (int stride = 1; stride < blockDim.x; stride *= 2) {
        int temp = 0;

        if (tid >= stride) {
            temp = sharedMemory[tid - stride];
        }

        __syncthreads();

        sharedMemory[tid] += temp;

        __syncthreads();
    }

    if (index < n) {
        output[index] = sharedMemory[tid];
    }
}

int main() {
    const int N = 8;

    int h_input[N] = {1,2,3,4,5,6,7,8};
    int h_output[N];

    int *d_input, *d_output;

    cudaMalloc(&d_input, N * sizeof(int));
    cudaMalloc(&d_output, N * sizeof(int));

    cudaMemcpy(
        d_input,
        h_input,
        N * sizeof(int),
        cudaMemcpyHostToDevice
    );

    PartialSumKernel<<<1, N, N * sizeof(int)>>>(
        d_input,
        d_output,
        N
    );

    cudaDeviceSynchronize();

    cudaMemcpy(
        h_output,
        d_output,
        N * sizeof(int),
        cudaMemcpyDeviceToHost
    );

    std::cout << "Input: ";
    for (int i = 0; i < N; i++) {
        std::cout << h_input[i] << " ";
    }

    std::cout << "\nPrefix Sum: ";
    for (int i = 0; i < N; i++) {
        std::cout << h_output[i] << " ";
    }

    std::cout << std::endl;

    cudaFree(d_input);
    cudaFree(d_output);

    return 0;
}

Writing prefix_sum.cu


In [5]:
!nvcc prefix_sum.cu -o prefix_sum

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [6]:
!./prefix_sum

Input: 1 2 3 4 5 6 7 8 
Prefix Sum: 1 3 6 10 15 21 28 36 
